# lltCapstoneGPT2 measurements (2x T4)

Throughput, peak memory and MFU for the 124M and 40M course GPT on FineWeb-Edu sample-10BT, DDP across both T4s, fp16 autocast + GradScaler, about 0.5M tokens per step.

In [ ]:

import json, os, platform, subprocess, sys, time
T_START = time.time()
RUN = {"module": "lltCapstoneGPT2", "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
       "probes": [], "benches": [], "log": {}}
os.makedirs("/kaggle/working/data", exist_ok=True)

def save():
    import torch
    RUN["env"] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    for mod in ("transformers", "datasets", "tiktoken", "numpy", "lm_eval"):
        try:
            RUN["env"][mod] = __import__(mod).__version__
        except Exception:
            pass
    RUN["elapsedSec"] = round(time.time() - T_START, 1)
    RUN["finished"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    with open("/kaggle/working/lltCapstoneGPT2_measure.json", "w") as f:
        json.dump(RUN, f, indent=1, default=str)

def sh(key, cmd, timeout=900, env=None):
    t = time.time()
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, env=env)
        out, rc = p.stdout, p.returncode
        if rc != 0:
            out += "\nSTDERR TAIL\n" + p.stderr[-3000:]
    except subprocess.TimeoutExpired as e:
        out, rc = "TIMEOUT " + str(e)[-500:], -9
    RUN["log"][key] = {"rc": rc, "sec": round(time.time() - t, 1), "out": out[-6000:]}
    print(key, "rc", rc, round(time.time() - t, 1), "s")
    print(out[-3000:])
    save()
    return out

def result_line(out):
    for line in out.splitlines():
        if line.startswith("RESULT "):
            return json.loads(line[7:])
        if line.startswith("{"):
            try:
                return json.loads(line)
            except Exception:
                pass
    return None

import torch
print(platform.python_version(), torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
try:
    import tiktoken
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
save()


In [ ]:
with open('gpt.py', 'w') as f:
    f.write('# gpt.py - the course GPT (lltGPT: RMSNorm, RoPE, SwiGLU, no biases, tied embeddings) at capstone sizes.\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\nCONFIGS = {\n    "124m": dict(V=50257, d=768, L=12, H=12, ctx=1024),   # GPT-2 small shape\n    "40m": dict(V=50257, d=384, L=12, H=6, ctx=1024),     # same depth, half the width\n}\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d, eps=1e-5):\n        super().__init__()\n        self.w = nn.Parameter(torch.ones(d))\n        self.eps = eps\n\n    def forward(self, x):\n        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)\n\n\ndef rope_tables(T, hd, device):\n    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device).float() / hd))\n    f = torch.outer(torch.arange(T, device=device).float(), inv)\n    return f.cos(), f.sin()\n\n\ndef apply_rope(x, cos, sin):\n    x1, x2 = x[..., 0::2], x[..., 1::2]\n    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)\n    return out.flatten(-2).type_as(x)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h):\n        super().__init__()\n        self.h = h\n        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)\n        self.qkv = nn.Linear(d, 3 * d, bias=False)\n        self.o = nn.Linear(d, d, bias=False)\n        hid = int(8 * d / 3 / 64 + 0.999) * 64\n        self.w1 = nn.Linear(d, hid, bias=False)\n        self.w3 = nn.Linear(d, hid, bias=False)\n        self.w2 = nn.Linear(hid, d, bias=False)\n\n    def forward(self, x, cos, sin):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.n1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))\n        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))\n        h = self.n2(x)\n        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))\n\n\nclass GPT(nn.Module):\n    def __init__(self, V, d, L, H, ctx, ckpt=False):\n        super().__init__()\n        self.emb = nn.Embedding(V, d)\n        self.blocks = nn.ModuleList([Block(d, H) for _ in range(L)])\n        self.norm = RMSNorm(d)\n        self.head = nn.Linear(d, V, bias=False)\n        self.head.weight = self.emb.weight            # tied, as in GPT-2\n        self.H, self.ctx, self.ckpt = H, ctx, ckpt\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                std = 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02\n                nn.init.normal_(p, 0.0, std)\n\n    def forward(self, idx, targets):\n        T = idx.shape[1]\n        cos, sin = rope_tables(T, self.emb.embedding_dim // self.H, idx.device)\n        x = self.emb(idx)\n        for b in self.blocks:\n            x = checkpoint(b, x, cos, sin, use_reentrant=False) if self.ckpt else b(x, cos, sin)\n        logits = self.head(self.norm(x))\n        return F.cross_entropy(logits.float().flatten(0, 1), targets.flatten())\n\n\ndef flops_per_token(cfg, n_params):\n    # nanoGPT / PaLM convention: 6N for the matmuls plus 12*L*d*T for attention scores and values\n    return 6 * n_params + 12 * cfg["L"] * cfg["d"] * cfg["ctx"]\n')
print('wrote', 'gpt.py')


In [ ]:
with open('prep_data.py', 'w') as f:
    f.write('# prep_data.py - stream FineWeb-Edu sample-10BT, tokenize with the GPT-2 tokenizer, write uint16 token files.\n# Usage: python prep_data.py <out_dir> <train_tokens> <val_tokens>\nimport json, sys, time\nimport numpy as np\nimport tiktoken\nfrom datasets import load_dataset\n\nout_dir, n_train, n_val = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])\nenc = tiktoken.get_encoding("gpt2")\nEOT = enc.eot_token                                    # 50256, put before every document as in llm.c\n\nt0 = time.time()\nds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)\nneed = n_train + n_val\nbuf, have, docs, chars = [], 0, 0, 0\nbatch = []\nfor row in ds:\n    batch.append(row["text"])\n    if len(batch) == 256:\n        for ids in enc.encode_ordinary_batch(batch, num_threads=4):\n            buf.append(np.array([EOT] + ids, dtype=np.uint16))\n            have += len(ids) + 1\n        docs += len(batch)\n        chars += sum(len(t) for t in batch)\n        batch = []\n        if have >= need:\n            break\ntoks = np.concatenate(buf)\nval, train = toks[:n_val], toks[n_val:n_val + n_train]   # first documents are the validation slice\ntrain.tofile(f"{out_dir}/train.bin")\nval.tofile(f"{out_dir}/val.bin")\ninfo = {"docs": docs, "chars": chars, "tokens": int(len(toks)), "train_tokens": int(len(train)),\n        "val_tokens": int(len(val)), "chars_per_token": round(chars / len(toks), 3), "sec": round(time.time() - t0, 1),\n        "tiktoken": tiktoken.__version__}\nprint(json.dumps(info))\n')
print('wrote', 'prep_data.py')


In [ ]:
with open('probe.py', 'w') as f:
    f.write('# probe.py - largest micro-batch that fits one T4 (one full train step, no DDP).\n# Usage: CUDA_VISIBLE_DEVICES=0 python probe.py <size> <ckpt 0|1> <b> [<b> ...]\nimport json, sys\nimport torch\nfrom gpt import GPT, CONFIGS\n\nsize, ckpt, sizes = sys.argv[1], sys.argv[2] == "1", [int(b) for b in sys.argv[3:]]\ncfg = CONFIGS[size]\ntorch.manual_seed(0)\nmodel = GPT(**cfg, ckpt=ckpt).cuda()\nopt = torch.optim.AdamW(model.parameters(), lr=6e-4, betas=(0.9, 0.95), weight_decay=0.1, fused=True)\nscaler = torch.amp.GradScaler("cuda")\nres = {}\nfor b in sizes:\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    try:\n        x = torch.randint(0, cfg["V"], (b, cfg["ctx"]), device="cuda")\n        with torch.autocast("cuda", dtype=torch.float16):\n            loss = model(x, x)\n        scaler.scale(loss).backward()\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n        torch.cuda.synchronize()\n        res[b] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n    except torch.cuda.OutOfMemoryError:\n        res[b] = "OOM"\n        opt.zero_grad(set_to_none=True)\n        break\nprint(json.dumps({"size": size, "ckpt": ckpt, "peakGB": res,\n                  "params": sum(p.numel() for p in model.parameters())}))\n')
print('wrote', 'probe.py')


In [ ]:
with open('bench.py', 'w') as f:
    f.write('# bench.py - sustained DDP training throughput on real FineWeb-Edu tokens, fp16 autocast + GradScaler.\n# Launch: torchrun --nproc_per_node=2 bench.py --size 124m --micro 12 [--compile] [--ckpt] [--resume-check]\nimport argparse, contextlib, datetime, json, os, random, time\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom gpt import GPT, CONFIGS, flops_per_token\n\np = argparse.ArgumentParser()\np.add_argument("--size", default="124m")\np.add_argument("--micro", type=int, default=8)\np.add_argument("--tokens-per-step", type=int, default=524288)\np.add_argument("--accum", type=int, default=0)          # 0 = derive from tokens-per-step\np.add_argument("--warmup", type=int, default=1)\np.add_argument("--steps", type=int, default=3)\np.add_argument("--compile", action="store_true")\np.add_argument("--ckpt", action="store_true")\np.add_argument("--resume-check", action="store_true")\np.add_argument("--data", default="/kaggle/working/data/train.bin")\np.add_argument("--tag", default="")\na = p.parse_args()\n\ndist.init_process_group("nccl", timeout=datetime.timedelta(minutes=5))\nrank, world = dist.get_rank(), dist.get_world_size()\ntorch.cuda.set_device(rank)\ncfg = CONFIGS[a.size]\nT = cfg["ctx"]\naccum = a.accum or max(1, round(a.tokens_per_step / (a.micro * T * world)))\ntokens_per_step = accum * a.micro * T * world\n\n\nclass Loader:\n    """Contiguous windows; rank r reads the r-th slice of each global micro-batch. pos is the resumable state."""\n    def __init__(self, path, b):\n        self.data = np.memmap(path, dtype=np.uint16, mode="r")\n        self.b, self.pos = b, 0\n\n    def next(self):\n        n = self.b * T\n        if self.pos + world * n + 1 > len(self.data):\n            self.pos = 0\n        s = self.pos + rank * n\n        w = torch.from_numpy(self.data[s:s + n + 1].astype(np.int64))\n        self.pos += world * n\n        x, y = w[:-1].view(self.b, T), w[1:].view(self.b, T)\n        return x.pin_memory().cuda(non_blocking=True), y.pin_memory().cuda(non_blocking=True)\n\n\ntorch.manual_seed(0)\nraw = GPT(**cfg, ckpt=a.ckpt).cuda()\nN = sum(q.numel() for q in raw.parameters())\nmodel = DDP(torch.compile(raw) if a.compile else raw, device_ids=[rank])\nopt = torch.optim.AdamW(raw.parameters(), lr=6e-4, betas=(0.9, 0.95), weight_decay=0.1, fused=True)\nscaler = torch.amp.GradScaler("cuda")\nloader = Loader(a.data, a.micro)\nstep = 0\n\n\ndef train_step():\n    global step\n    lr = 6e-4 * min(1.0, (step + 1) / 10)\n    for g in opt.param_groups:\n        g["lr"] = lr\n    tot = torch.zeros((), device="cuda")\n    for i in range(accum):\n        x, y = loader.next()\n        ctx = contextlib.nullcontext() if i == accum - 1 else model.no_sync()\n        with ctx:\n            with torch.autocast("cuda", dtype=torch.float16):\n                loss = model(x, y) / accum\n            scaler.scale(loss).backward()\n        tot += loss.detach()\n    scaler.unscale_(opt)\n    gn = torch.nn.utils.clip_grad_norm_(raw.parameters(), 1.0)\n    scaler.step(opt)\n    scaler.update()\n    opt.zero_grad(set_to_none=True)\n    dist.all_reduce(tot, op=dist.ReduceOp.AVG)\n    step += 1\n    return tot.item(), gn.item()\n\n\nrec = {"tag": a.tag, "size": a.size, "params": N, "micro": a.micro, "accum": accum, "world": world,\n       "tokensPerStep": tokens_per_step, "compile": a.compile, "ckpt": a.ckpt, "steps": []}\ntry:\n    t_first = time.perf_counter()\n    for _ in range(a.warmup):\n        torch.cuda.synchronize()\n        t0 = time.perf_counter()\n        loss, gn = train_step()\n        torch.cuda.synchronize()\n        rec["steps"].append({"step": step, "sec": round(time.perf_counter() - t0, 2), "loss": round(loss, 4), "warmup": True})\n    rec["warmupSec"] = round(time.perf_counter() - t_first, 1)\n    torch.cuda.reset_peak_memory_stats()\n    t_all = time.perf_counter()\n    for _ in range(a.steps):\n        torch.cuda.synchronize()\n        t0 = time.perf_counter()\n        loss, gn = train_step()\n        torch.cuda.synchronize()\n        rec["steps"].append({"step": step, "sec": round(time.perf_counter() - t0, 2), "loss": round(loss, 4),\n                             "gradNorm": round(gn, 3), "scale": scaler.get_scale()})\n    dt = time.perf_counter() - t_all\n    tps = a.steps * tokens_per_step / dt\n    peak = torch.tensor([torch.cuda.max_memory_allocated() / 1e9, torch.cuda.max_memory_reserved() / 1e9], device="cuda")\n    dist.all_reduce(peak, op=dist.ReduceOp.MAX)\n    fpt = flops_per_token(cfg, N)\n    rec.update({"tokPerSec": round(tps), "secPerStep": round(dt / a.steps, 2), "peakAllocGB": round(peak[0].item(), 2),\n                "peakReservedGB": round(peak[1].item(), 2), "flopsPerToken": fpt,\n                "achievedTFLOPS": round(tps * fpt / 1e12, 2),\n                "mfuVsSpec65": round(tps * fpt / (world * 65e12), 4),\n                "mfuVsMeasured22p1": round(tps * fpt / (world * 22.1e12), 4)})\nexcept torch.cuda.OutOfMemoryError:\n    rec["oom"] = True\n    rec["allocAtOomGB"] = round(torch.cuda.memory_allocated() / 1e9, 2)\n\nif a.resume_check and not rec.get("oom"):\n    # Exact resume: save full state, take 2 steps, reload into the live objects, take the same 2 steps again.\n    path = "/tmp/capstone_ckpt.pt"\n    t0 = time.perf_counter()\n    if rank == 0:\n        state = {"model": raw.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),\n                 "loader_pos": loader.pos, "step": step,\n                 "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),\n                         "cuda": torch.cuda.get_rng_state_all()}}\n        torch.save(state, path + ".tmp")\n        os.replace(path + ".tmp", path)\n    dist.barrier()\n    save_sec = time.perf_counter() - t0\n    cont = [train_step()[0] for _ in range(2)]\n    t0 = time.perf_counter()\n    st = torch.load(path, map_location=f"cuda:{rank}", weights_only=False)\n    raw.load_state_dict(st["model"])\n    opt.load_state_dict(st["optimizer"])\n    scaler.load_state_dict(st["scaler"])\n    loader.pos, step = st["loader_pos"], st["step"]\n    random.setstate(st["rng"]["python"])\n    np.random.set_state(st["rng"]["numpy"])\n    torch.set_rng_state(st["rng"]["torch"])\n    dist.barrier()\n    load_sec = time.perf_counter() - t0\n    again = [train_step()[0] for _ in range(2)]\n    rec["resume"] = {"fileGB": round(os.path.getsize(path) / 1e9, 3), "saveSec": round(save_sec, 1),\n                     "loadSec": round(load_sec, 1), "continued": cont, "reloaded": again,\n                     "maxAbsDiff": max(abs(u - v) for u, v in zip(cont, again))}\n\nif rank == 0:\n    print("RESULT " + json.dumps(rec))\ndist.destroy_process_group()\n')
print('wrote', 'bench.py')


In [ ]:

out = sh("prep", [sys.executable, "prep_data.py", "/kaggle/working/data", "6000000", "500000"], timeout=900)
RUN["data"] = result_line(out)
save()


In [ ]:

env1 = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
for key, args in [("probe_124m", ["124m", "0", "8", "12", "16"]),
                  ("probe_124m_ckpt", ["124m", "1", "16", "24", "32"]),
                  ("probe_40m", ["40m", "0", "16", "24", "32", "48"])]:
    r = result_line(sh(key, [sys.executable, "probe.py"] + args, timeout=300, env=env1))
    RUN["probes"].append(r)
save()

def pick(r, ddp_extra_gb):
    ok = [int(b) for b, v in r["peakGB"].items() if v != "OOM" and v + ddp_extra_gb < 14.8]
    return max(ok) if ok else None

P = {r["size"] + ("_ckpt" if r["ckpt"] else ""): r for r in RUN["probes"] if r}
B124 = pick(P["124m"], 4 * P["124m"]["params"] / 1e9 + 0.3)
B124C = pick(P["124m_ckpt"], 4 * P["124m"]["params"] / 1e9 + 0.3)
B40 = pick(P["40m"], 4 * P["40m"]["params"] / 1e9 + 0.3)
RUN["picked"] = {"124m": B124, "124m_ckpt": B124C, "40m": B40}
print(RUN["picked"])
save()


In [ ]:
r = result_line(sh('124m_eager', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '124m_eager'] + ['--size', '124m', '--micro', str(B124), '--warmup', '1', '--steps', '3'], timeout=900))
RUN['benches'].append(r)
save()


In [ ]:
r = result_line(sh('124m_compile', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '124m_compile'] + ['--size', '124m', '--micro', str(B124), '--warmup', '2', '--steps', '3', '--compile'], timeout=900))
RUN['benches'].append(r)
save()


In [ ]:
r = result_line(sh('40m_eager', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '40m_eager'] + ['--size', '40m', '--micro', str(B40), '--warmup', '1', '--steps', '4'], timeout=900))
RUN['benches'].append(r)
save()


In [ ]:
r = result_line(sh('40m_compile', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '40m_compile'] + ['--size', '40m', '--micro', str(B40), '--warmup', '2', '--steps', '4', '--compile'], timeout=900))
RUN['benches'].append(r)
save()


In [ ]:
if time.time() - T_START < 15 * 60:
    r = result_line(sh('124m_ckpt_eager', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '124m_ckpt_eager'] + ['--size', '124m', '--micro', str(B124C), '--warmup', '1', '--steps', '2', '--ckpt'], timeout=900))
    RUN['benches'].append(r)
    save()


In [ ]:
if time.time() - T_START < 18 * 60:
    r = result_line(sh('124m_resume', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '124m_resume'] + ['--size', '124m', '--micro', str(B124), '--accum', '2', '--warmup', '1', '--steps', '1', '--resume-check'], timeout=900))
    RUN['benches'].append(r)
    save()


In [ ]:
if time.time() - T_START < 20 * 60:
    r = result_line(sh('124m_ckpt_compile', [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'bench.py', '--tag', '124m_ckpt_compile'] + ['--size', '124m', '--micro', str(B124C), '--warmup', '2', '--steps', '2', '--ckpt', '--compile'], timeout=900))
    RUN['benches'].append(r)
    save()


In [ ]:

if time.time() - T_START < 26 * 60:
    sh("pip_lmeval", [sys.executable, "-m", "pip", "install", "-q", "lm-eval==0.4.13"], timeout=300)
    env1 = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
    sh("lmeval_gpt2", ["lm_eval", "--model", "hf", "--model_args", "pretrained=openai-community/gpt2,dtype=float16",
                       "--tasks", "hellaswag,arc_easy,piqa,lambada_openai", "--device", "cuda:0", "--batch_size", "64",
                       "--output_path", "/kaggle/working/lmeval"], timeout=900, env=env1)
else:
    RUN["log"]["lmeval_gpt2"] = "skipped: time budget"
save()
import shutil
shutil.rmtree("/kaggle/working/data", ignore_errors=True)
print("total min", round((time.time() - T_START) / 60, 1))
save()
